# Assignment 04 — 01: Supervised Fine-Tuning (5 LoRA trials)
**Track 1 / Option A** | Base: `Qwen/Qwen3-0.6B-Base`

Group: **Abdullah Iqbal (26904), Anushe Ali (26418)**

Runs 5 SFT+LoRA trials, evaluates each on the 10 prompts (BLEU+BERTScore),
and selects the best (tie-break = lower validation loss).

## 1. Install & mount

In [1]:
# Run once per Colab session. Restart runtime if prompted after install.
!pip install -q -U "transformers>=4.51" "trl>=0.15" "peft>=0.11" \
    "datasets>=2.19" accelerate sacrebleu bert-score matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJ = '/content/drive/MyDrive/assignment-4'   # change if you like
os.makedirs(PROJ + '/results', exist_ok=True)
os.makedirs(PROJ + '/adapters', exist_ok=True)
print('Saving outputs to', PROJ)

Mounted at /content/drive
Saving outputs to /content/drive/MyDrive/assignment-4


In [3]:
import json, torch
PROMPT_TEMPLATE = '### Instruction:\n{instruction}\n\n### Response:\n'
def format_prompt(instruction):
    return PROMPT_TEMPLATE.format(instruction=instruction.strip())
def pick_dtype():
    if torch.cuda.is_available():
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    return torch.float32

@torch.no_grad()
def generate_response(model, tokenizer, instruction, max_new_tokens=256):
    prompt = format_prompt(instruction)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    gen = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def generate_all(model, tokenizer, test_set, max_new_tokens=256):
    rows = []
    for ex in test_set:
        rows.append({'id': ex['id'], 'instruction': ex['instruction'],
                     'reference': ex['reference'],
                     'response': generate_response(model, tokenizer, ex['instruction'], max_new_tokens)})
    return rows

def compute_bleu(hyps, refs):
    import sacrebleu
    s = [sacrebleu.sentence_bleu(h, [r]).score for h, r in zip(hyps, refs)]
    return sum(s) / max(len(s), 1)

def compute_bertscore(hyps, refs, model_type='roberta-large'):
    from bert_score import score as bert_score
    P, R, F1 = bert_score(hyps, refs, lang='en', model_type=model_type, verbose=False)
    return float(F1.mean())

def evaluate_rows(rows, bertscore_model='roberta-large'):
    hyps = [r['response'] for r in rows]; refs = [r['reference'] for r in rows]
    bleu = compute_bleu(hyps, refs); bert = compute_bertscore(hyps, refs, bertscore_model)
    return {'bleu': bleu, 'bertscore_f1': bert, 'composite': 0.5*(bleu/100.0)+0.5*bert}

def select_best(trials, tol=0.005):
    ranked = sorted(trials, key=lambda t: t['composite'], reverse=True)
    top = ranked[0]['composite']
    cont = [t for t in ranked if top - t['composite'] <= tol]
    if len(cont) > 1:
        cont = sorted(cont, key=lambda t: t.get('val_loss', float('inf')))
    return cont[0]

In [4]:
# Upload data/test_set.json to PROJ (or to Colab and adjust path).
# IMPORTANT: fill the 'reference' fields with gold answers from ChatGPT/Claude/Gemini first!
TEST_PATH = PROJ + '/data/test_set.json'
with open(TEST_PATH) as f:
    test_set = json.load(f)
assert all('<<PASTE' not in ex['reference'] for ex in test_set), \
    'Fill in the gold reference answers in test_set.json before running!'
print(len(test_set), 'test prompts loaded')

10 test prompts loaded


## 2. Load and preprocess the SFT dataset
Dataset: `databricks/databricks-dolly-15k` (instruction-response, not used in class).
We take a subset for the Colab compute budget and format each example with our prompt
template. Adjust `N_SAMPLES` to trade speed vs quality.

In [5]:
MODEL_ID = 'Qwen/Qwen3-0.6B-Base'
from datasets import load_dataset
N_SAMPLES = 3000   # justify this subset size in the report
raw = load_dataset('databricks/databricks-dolly-15k', split='train')
# Keep examples without a separate context for a clean instruction->response signal.
raw = raw.filter(lambda x: len(x['context'].strip()) == 0)
raw = raw.shuffle(seed=42).select(range(min(N_SAMPLES, len(raw))))
def to_text(ex):
    return {'text': format_prompt(ex['instruction']) + ex['response']}
ds = raw.map(to_text, remove_columns=raw.column_names)
ds = ds.train_test_split(test_size=0.1, seed=42)
train_ds, val_ds = ds['train'], ds['test']
print(train_ds, '\nExample:\n', train_ds[0]['text'][:300])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Filter:   0%|          | 0/15011 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 2700
}) 
Example:
 ### Instruction:
Does Delta Lake support multi-table transactions?

### Response:
Delta Lake does not support multi-table transactions and foreign keys. Delta Lake supports transactions at the table level.


## 3. Define the 5 trials

In [6]:
# 5 SFT trials spanning a wide range of LoRA + training configs.
# 'batch' = per-device batch (kept small for the 16GB T4); 'grad_accum' multiplies
# it to reach the EFFECTIVE batch size = batch * grad_accum (what we report/vary).
SFT_TRIALS = [
    dict(trial=1, r=8,  alpha=16, modules=['q_proj','v_proj'],
         lr=2e-4, batch=8, grad_accum=1, epochs=1),                       # eff batch 8
    dict(trial=2, r=16, alpha=32, modules=['q_proj','k_proj','v_proj','o_proj'],
         lr=2e-4, batch=8, grad_accum=1, epochs=2),                       # eff batch 8
    dict(trial=3, r=32, alpha=64, modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
         lr=1e-4, batch=4, grad_accum=1, epochs=2),                       # eff batch 4
    dict(trial=4, r=16, alpha=32, modules=['q_proj','v_proj'],
         lr=3e-4, batch=4, grad_accum=4, epochs=3),                       # eff batch 16
    dict(trial=5, r=64, alpha=128, modules='all-linear',
         lr=5e-5, batch=2, grad_accum=2, epochs=1),                       # eff batch 4
]

In [7]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## 4. Training loop
Each trial loads a fresh copy of the base model, attaches LoRA, trains, records
validation loss, saves the adapter, then evaluates on the 10 prompts.

In [8]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import LoraConfig
# from trl import SFTTrainer, SFTConfig
# import gc, json

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
# dtype = pick_dtype()
# use_bf16 = dtype == torch.bfloat16

# sft_results = []
# for cfg in SFT_TRIALS:
#     print('\n==== SFT TRIAL', cfg['trial'], cfg, '====')
#     model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map='auto')
#     model.config.use_cache = False
#     lora = LoraConfig(r=cfg['r'], lora_alpha=cfg['alpha'], lora_dropout=0.05,
#                       target_modules=cfg['modules'], task_type='CAUSAL_LM')
#     adapter_dir = PROJ + f"/adapters/sft_trial{cfg['trial']}"
#     args = SFTConfig(output_dir=adapter_dir, num_train_epochs=cfg['epochs'],
#         per_device_train_batch_size=cfg['batch'], per_device_eval_batch_size=cfg['batch'],
#         gradient_accumulation_steps=1, learning_rate=cfg['lr'], logging_steps=25,
#         eval_strategy='epoch', save_strategy='no', max_length=512,
#         dataset_text_field='text', report_to='none',
#         bf16=use_bf16, fp16=not use_bf16)
#     trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
#                          eval_dataset=val_ds, peft_config=lora, processing_class=tokenizer)
#     trainer.train()
#     val_loss = trainer.evaluate()['eval_loss']
#     trainer.save_model(adapter_dir)
#     # Evaluate on the 10 prompts
#     model.config.use_cache = True; model.eval()
#     rows = generate_all(model, tokenizer, test_set)
#     metrics = evaluate_rows(rows)
#     rec = dict(trial=cfg['trial'], **metrics, val_loss=val_loss,
#                config={k: (list(v) if isinstance(v, list) else v) for k, v in cfg.items()},
#                rows=rows, adapter_dir=adapter_dir)
#     sft_results.append(rec)
#     print('Trial %d  BLEU=%.2f  BERT=%.4f  composite=%.4f  val_loss=%.4f' %
#           (cfg['trial'], metrics['bleu'], metrics['bertscore_f1'], metrics['composite'], val_loss))
#     del model, trainer; gc.collect(); torch.cuda.empty_cache()

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
import gc, json, os

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
dtype = pick_dtype()
use_bf16 = dtype == torch.bfloat16

def eval_on_prompts(model):
    model.config.use_cache = True; model.eval()
    rows = generate_all(model, tokenizer, test_set)
    return rows, evaluate_rows(rows)

sft_results = []
for cfg in SFT_TRIALS:
    print('\n==== SFT TRIAL', cfg['trial'], cfg, '====')
    adapter_dir = PROJ + f"/adapters/sft_trial{cfg['trial']}"
    res_path = PROJ + f"/results/sft_trial{cfg['trial']}.json"
    # ---- Resume: already fully evaluated -> just load the saved result.
    if os.path.exists(res_path):
        sft_results.append(json.load(open(res_path)))
        print('  [resume] loaded saved result, skipping.'); continue
    cfgj = {k: (list(v) if isinstance(v, list) else v) for k, v in cfg.items()}
    # ---- Resume: adapter exists but no result -> evaluate without retraining.
    if os.path.isdir(adapter_dir) and os.path.exists(adapter_dir + '/adapter_config.json'):
        print('  [resume] adapter found, evaluating without retraining.')
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map='auto')
        m = PeftModel.from_pretrained(m, adapter_dir)
        rows, metrics = eval_on_prompts(m)
        rec = dict(trial=cfg['trial'], **metrics, val_loss=None, config=cfgj,
                   rows=rows, adapter_dir=adapter_dir)
        del m; gc.collect(); torch.cuda.empty_cache()
    else:
        # ---- Train from scratch.
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map='auto')
        model.config.use_cache = False
        lora = LoraConfig(r=cfg['r'], lora_alpha=cfg['alpha'], lora_dropout=0.05,
                          target_modules=cfg['modules'], task_type='CAUSAL_LM')
        args = SFTConfig(output_dir=adapter_dir, num_train_epochs=cfg['epochs'],
            per_device_train_batch_size=cfg['batch'], per_device_eval_batch_size=4,
            gradient_accumulation_steps=cfg['grad_accum'], learning_rate=cfg['lr'],
            logging_steps=25, eval_strategy='epoch', save_strategy='no', max_length=512,
            dataset_text_field='text', report_to='none',
            gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
            bf16=use_bf16, fp16=not use_bf16)
        trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                             eval_dataset=val_ds, peft_config=lora, processing_class=tokenizer)
        trainer.train()
        val_loss = trainer.evaluate()['eval_loss']
        trainer.save_model(adapter_dir)
        model.gradient_checkpointing_disable()
        rows, metrics = eval_on_prompts(model)
        rec = dict(trial=cfg['trial'], **metrics, val_loss=val_loss, config=cfgj,
                   rows=rows, adapter_dir=adapter_dir)
        del model, trainer; gc.collect(); torch.cuda.empty_cache()
    json.dump(rec, open(res_path, 'w'), indent=2)   # checkpoint this trial
    sft_results.append(rec)
    vl = rec['val_loss'];  vl_s = ('%.4f' % vl) if vl is not None else 'n/a'
    print('Trial %d  BLEU=%.2f  BERT=%.4f  composite=%.4f  val_loss=%s' %
          (cfg['trial'], metrics['bleu'], metrics['bertscore_f1'], metrics['composite'], vl_s))

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]


==== SFT TRIAL 1 {'trial': 1, 'r': 8, 'alpha': 16, 'modules': ['q_proj', 'v_proj'], 'lr': 0.0002, 'batch': 8, 'grad_accum': 1, 'epochs': 1} ====
  [resume] loaded saved result, skipping.

==== SFT TRIAL 2 {'trial': 2, 'r': 16, 'alpha': 32, 'modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj'], 'lr': 0.0002, 'batch': 8, 'grad_accum': 1, 'epochs': 2} ====
  [resume] loaded saved result, skipping.

==== SFT TRIAL 3 {'trial': 3, 'r': 32, 'alpha': 64, 'modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], 'lr': 0.0001, 'batch': 4, 'grad_accum': 1, 'epochs': 2} ====
  [resume] loaded saved result, skipping.

==== SFT TRIAL 4 {'trial': 4, 'r': 16, 'alpha': 32, 'modules': ['q_proj', 'v_proj'], 'lr': 0.0003, 'batch': 4, 'grad_accum': 4, 'epochs': 3} ====
  [resume] loaded saved result, skipping.

==== SFT TRIAL 5 {'trial': 5, 'r': 64, 'alpha': 128, 'modules': 'all-linear', 'lr': 5e-05, 'batch': 2, 'grad_accum': 2, 'epochs': 1} ====
  [resume] loaded saved resul

## 5. Results table + best-model selection

In [10]:
import pandas as pd
df = pd.DataFrame([{k: r[k] for k in ['trial','bleu','bertscore_f1','composite','val_loss']}
                   for r in sft_results])
display(df)
best = select_best(sft_results)
print('BEST SFT TRIAL =', best['trial'], '| config:', best['config'])
with open(PROJ + '/results/sft_trials.json', 'w') as f:
    json.dump({'trials': [{k:v for k,v in r.items()} for r in sft_results],
               'best_trial': best['trial'], 'best_adapter': best['adapter_dir']}, f, indent=2)
print('Saved sft_trials.json. Best adapter:', best['adapter_dir'])

,trial,bleu,bertscore_f1,composite,val_loss
0,1,6.115311,0.875189,0.468171,NaN
1,2,6.116716,0.878470,0.469818,NaN
2,3,9.465842,0.883640,0.489149,NaN
3,4,6.565772,0.875503,0.470580,2.125044
4,5,5.770018,0.872104,0.464902,2.107723


BEST SFT TRIAL = 3 | config: {'trial': 3, 'r': 32, 'alpha': 64, 'modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], 'lr': 0.0001, 'batch': 4, 'epochs': 2}
Saved sft_trials.json. Best adapter: /content/drive/MyDrive/assignment-4/adapters/sft_trial3


### Report this
- BLEU + BERTScore for **each** trial (Table).
- Which trial won and **why** (composite score; tie-break val loss).
- Effect of LoRA rank / target modules / LR on output quality.